# **Import Modules**

In [108]:
import numpy as np 
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, cross_val_score 
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, FunctionTransformer, StandardScaler, LabelEncoder 
from sklearn.compose import ColumnTransformer 
from sklearn.pipeline import Pipeline 
from sklearn.metrics import accuracy_score, confusion_matrix, roc_curve, auc, roc_auc_score, classification_report

from sklearn.tree import DecisionTreeClassifier 
from sklearn.ensemble import RandomForestClassifier 

import matplotlib.pyplot as plt 

import seaborn as sns

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

import warnings
warnings.filterwarnings("ignore")

# **Load Dataset**

In [42]:
train = pd.read_csv('train.csv')

test = pd.read_csv('test.csv')

# **Data Understanding**

In [43]:
train.head()

,id,Gender,Age,Height,Weight,family_history_with_overweight,FAVC,FCVC,NCP,CAEC,SMOKE,CH2O,SCC,FAF,TUE,CALC,MTRANS,NObeyesdad
0,0,Male,24.443011,1.699998,81.669950,yes,yes,2.000000,2.983297,Sometimes,no,2.763573,no,0.000000,0.976473,Sometimes,Public_Transportation,Overweight_Level_II
1,1,Female,18.000000,1.560000,57.000000,yes,yes,2.000000,3.000000,Frequently,no,2.000000,no,1.000000,1.000000,no,Automobile,Normal_Weight
2,2,Female,18.000000,1.711460,50.165754,yes,yes,1.880534,1.411685,Sometimes,no,1.910378,no,0.866045,1.673584,no,Public_Transportation,Insufficient_Weight
3,3,Female,20.952737,1.710730,131.274851,yes,yes,3.000000,3.000000,Sometimes,no,1.674061,no,1.467863,0.780199,Sometimes,Public_Transportation,Obesity_Type_III
4,4,Male,31.641081,1.914186,93.798055,yes,yes,2.679664,1.971472,Sometimes,no,1.979848,no,1.967973,0.931721,Sometimes,Public_Transportation,Overweight_Level_II


In [44]:
test.head()

,id,Gender,Age,Height,Weight,family_history_with_overweight,FAVC,FCVC,NCP,CAEC,SMOKE,CH2O,SCC,FAF,TUE,CALC,MTRANS
0,20758,Male,26.899886,1.848294,120.644178,yes,yes,2.938616,3.000000,Sometimes,no,2.825629,no,0.855400,0.000000,Sometimes,Public_Transportation
1,20759,Female,21.000000,1.600000,66.000000,yes,yes,2.000000,1.000000,Sometimes,no,3.000000,no,1.000000,0.000000,Sometimes,Public_Transportation
2,20760,Female,26.000000,1.643355,111.600553,yes,yes,3.000000,3.000000,Sometimes,no,2.621877,no,0.000000,0.250502,Sometimes,Public_Transportation
3,20761,Male,20.979254,1.553127,103.669116,yes,yes,2.000000,2.977909,Sometimes,no,2.786417,no,0.094851,0.000000,Sometimes,Public_Transportation
4,20762,Female,26.000000,1.627396,104.835346,yes,yes,3.000000,3.000000,Sometimes,no,2.653531,no,0.000000,0.741069,Sometimes,Public_Transportation


In [45]:
train.shape

(20758, 18)

In [46]:
test.shape

(13840, 17)

In [47]:
train['NObeyesdad'].value_counts()

NObeyesdad
Obesity_Type_III       4046
Obesity_Type_II        3248
Normal_Weight          3082
Obesity_Type_I         2910
Insufficient_Weight    2523
Overweight_Level_II    2522
Overweight_Level_I     2427
Name: count, dtype: int64

In [48]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20758 entries, 0 to 20757
Data columns (total 18 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   id                              20758 non-null  int64  
 1   Gender                          20758 non-null  object 
 2   Age                             20758 non-null  float64
 3   Height                          20758 non-null  float64
 4   Weight                          20758 non-null  float64
 5   family_history_with_overweight  20758 non-null  object 
 6   FAVC                            20758 non-null  object 
 7   FCVC                            20758 non-null  float64
 8   NCP                             20758 non-null  float64
 9   CAEC                            20758 non-null  object 
 10  SMOKE                           20758 non-null  object 
 11  CH2O                            20758 non-null  float64
 12  SCC                             

In [49]:
train.duplicated().sum()

0

In [50]:
train['Gender'].value_counts()

Gender
Female    10422
Male      10336
Name: count, dtype: int64

In [51]:
train['family_history_with_overweight'].value_counts()

family_history_with_overweight
yes    17014
no      3744
Name: count, dtype: int64

In [52]:
train['CAEC'].value_counts()

CAEC
Sometimes     17529
Frequently     2472
Always          478
no              279
Name: count, dtype: int64

In [53]:
train['SMOKE'].value_counts()

SMOKE
no     20513
yes      245
Name: count, dtype: int64

In [54]:
train['CALC'].value_counts()

CALC
Sometimes     15066
no             5163
Frequently      529
Name: count, dtype: int64

In [55]:
train['SCC'].value_counts()

SCC
no     20071
yes      687
Name: count, dtype: int64

In [56]:
train['FAVC'].value_counts()

FAVC
yes    18982
no      1776
Name: count, dtype: int64

# **Feature Engineering**

In [57]:
train.columns

Index(['id', 'Gender', 'Age', 'Height', 'Weight',
       'family_history_with_overweight', 'FAVC', 'FCVC', 'NCP', 'CAEC',
       'SMOKE', 'CH2O', 'SCC', 'FAF', 'TUE', 'CALC', 'MTRANS', 'NObeyesdad'],
      dtype='object')

In [58]:
bin_columns = ["Gender", "family_history_with_overweight", "FAVC", "SMOKE", "SCC"]

num_columns = ["Age", "Height", "Weight", "FCVC", "NCP", "CH2O", "FAF", "TUE"]

cat_columns = ["CAEC", "CALC", "MTRANS"]

# Encode Target Columns

In [59]:
le = LabelEncoder()

y = le.fit_transform(train['NObeyesdad'])

In [60]:
X = train.drop(columns=['id', 'NObeyesdad'])

X_test_final = test.drop(columns=['id'])

In [61]:
def map_yes_no_gender(df):
    out = df.copy()

    mapping_yes_no = {"yes": 1, "no": 0}
    mapping_gender = {"Male": 1, "Female": 0}

    for col in out.columns:  # iterate only over the subset
        if col == "Gender":
            out[col] = out[col].map(mapping_gender)
        else:  # the other 4 are yes/no
            out[col] = out[col].map(mapping_yes_no)

    return out


# **Data Preprocessing**

In [62]:
yn_mapper = FunctionTransformer(map_yes_no_gender, feature_names_out="one-to-one")

In [63]:
binary_pipe = Pipeline([
    ('yn_map', yn_mapper)
])

numeric_pipe = Pipeline([
    ('scalar', StandardScaler())
])

cat_pipe = Pipeline([
    ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))    
])

In [64]:
preprocessor = ColumnTransformer(
    transformers= [
        ('bin', binary_pipe, bin_columns),
        ('num', numeric_pipe, num_columns),
        ('cat', cat_pipe, cat_columns)
    ], 
    remainder='drop'
)

# **Train-Test Split**

In [65]:
# Train-Test Split

X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42)

In [66]:
# Helper Functions

def run_search_grid(pipe, params, name):

    grid = GridSearchCV(pipe, params, cv=5, n_jobs=-1, scoring='accuracy')

    grid.fit(X_train, y_train)

    print(f"\n✅ Finished GridSearch for {name}")
    print("Best Params:", grid.best_params_)

    return grid


def run_search_rand(pipe, params_dist, name):

    rand = RandomizedSearchCV(pipe, params_dist, cv=5, n_jobs=-1, scoring='accuracy')

    rand.fit(X_train, y_train)

    print(f'\n Finished Randomized Search for {name}')
    print('Best Params:', rand.best_estimator_)

    return rand

In [98]:

def evaluate_model(grid, model_name, X, y, X_valid, y_valid):
    best_model = grid.best_estimator_
    val_score = cross_val_score(best_model, X, y, cv=5).mean()
    test_score = accuracy_score(y_valid, best_model.predict(X_valid))
    gap = val_score - test_score
    
    if gap > 0.05 and val_score >= 0.85:
        fit_msg = "🚨 Overfitting"
    elif val_score < 0.70 and test_score < 0.70:
        fit_msg = "⚠️ Underfitting"
    elif abs(gap) <= 0.05 and test_score >= 0.75:
        fit_msg = "✅ Good Fit"
    else:
        fit_msg = "ℹ️ Borderline"
    
    print(f"\n--- {model_name} ---")
    print("Validation Accuracy:", round(val_score, 4))
    print("Test Accuracy:", round(test_score, 4))
    print("Gap:", round(gap, 4))
    print("Fit Check:", fit_msg)
    
    return {
        "Model": model_name,
        "Val_Accuracy": round(val_score, 4),
        "Test_Accuracy": round(test_score, 4),
        "Gap": round(gap, 4),
        "Fit": fit_msg
    }

# Model Building and Evaluation

# **Random Forest**

In [67]:
rf_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(random_state=42))
])

In [68]:
rf_pipe.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('bin', ...), ('num', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [69]:
rf_pipe.score(X_valid, y_valid)

0.890655105973025

In [70]:
y_pred = rf_pipe.predict(X_valid)

accuracy_score(y_valid, y_pred)

0.890655105973025

# **Decision Tree**

In [78]:
dt_pipe = Pipeline([
    ('preprocessor', preprocessor), 
    ('model', DecisionTreeClassifier(random_state=42))
])

In [79]:
dt_pipe.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('bin', ...), ('num', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [80]:
dt_pipe.score(X_valid, y_valid)

0.8463391136801541

# **XGBoost**

In [82]:
xgb_clf = XGBClassifier(use_label_encoder=False, eval_metric="mlogloss", random_state=42)

xgb_pipe = Pipeline([
    ("pre", preprocessor),
    ("clf", xgb_clf)
])

In [83]:
xgb_pipe.fit(X_train, y_train)

d:\anaconda3\Lib\site-packages\xgboost\training.py:183: UserWarning: [13:05:50] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


,steps,"[('pre', ...), ('clf', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('bin', ...), ('num', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [84]:
xgb_pipe.score(X_valid, y_valid)

0.9019749518304432

# **LightGBM**

In [85]:
lgbm_clf = LGBMClassifier(random_state=42)

lgbm_pipe = Pipeline([
    ("pre", preprocessor),
    ("clf", lgbm_clf)
])

In [86]:
lgbm_pipe.fit(X_train, y_train)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.010480 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2031
[LightGBM] [Info] Number of data points in the train set: 16606, number of used features: 25
[LightGBM] [Info] Start training from score -2.117117
[LightGBM] [Info] Start training from score -1.911230
[LightGBM] [Info] Start training from score -1.948141
[LightGBM] [Info] Start training from score -1.857720
[LightGBM] [Info] Start training from score -1.633574
[LightGBM] [Info] Start training from score -2.145531
[LightGBM] [Info] Start training from score -2.112625


,steps,"[('pre', ...), ('clf', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('bin', ...), ('num', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [87]:
lgbm_pipe.score(X_valid, y_valid)

d:\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


0.9039017341040463

In [102]:
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score
import pandas as pd

# Define models inside pipelines
models = {
    "Random Forest": Pipeline([
        ("preprocessor", preprocessor),
        ("clf", RandomForestClassifier(random_state=42))
    ]),
    "Decision Tree": Pipeline([
        ("preprocessor", preprocessor),
        ("clf", DecisionTreeClassifier(random_state=42))
    ]),
    "XGBoost": Pipeline([
        ("preprocessor", preprocessor),
        ("clf", XGBClassifier(use_label_encoder=False, eval_metric="mlogloss", random_state=42))
    ]),
    "LightGBM": Pipeline([
        ("preprocessor", preprocessor),
        ("clf", LGBMClassifier(random_state=42))
    ])
}

results = []

# Loop through models
for name, pipe in models.items():
    # Fit on training data
    pipe.fit(X_train, y_train)

    # Validation accuracy (hold-out set)
    y_pred = pipe.predict(X_valid)
    acc = accuracy_score(y_valid, y_pred)

    # Cross-validation (on full dataset X, y)
    cv_scores = cross_val_score(pipe, X, y, cv=5, scoring="accuracy", n_jobs=-1)
    cv_mean = cv_scores.mean()

    # Compare CV vs Validation accuracy → detect fit status
    if acc < cv_mean - 0.02:  
        fit_status = "Underfitting"
    elif acc > cv_mean + 0.02:  
        fit_status = "Overfitting"
    else:  
        fit_status = "Good Fit"

    # Store results
    results.append({
        "Model": name,
        "CrossValScore": round(cv_mean, 4),
        "AccuracyScore": round(acc, 4),
        "FitStatus": fit_status
    })

# Convert to DataFrame
results_df = pd.DataFrame(results)
print(results_df)


d:\anaconda3\Lib\site-packages\xgboost\training.py:183: UserWarning: [13:45:20] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001328 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2031
[LightGBM] [Info] Number of data points in the train set: 16606, number of used features: 25
[LightGBM] [Info] Start training from score -2.117117
[LightGBM] [Info] Start training from score -1.911230
[LightGBM] [Info] Start training from score -1.948141
[LightGBM] [Info] Start training from score -1.857720
[LightGBM] [Info] Start training from score -1.633574
[LightGBM] [Info] Start training from score -2.145531
[LightGBM] [Info] Start training from score -2.112625


d:\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


           Model  CrossValScore  AccuracyScore FitStatus
0  Random Forest         0.8938         0.8907  Good Fit
1  Decision Tree         0.8443         0.8463  Good Fit
2        XGBoost         0.9046         0.9020  Good Fit
3       LightGBM         0.9045         0.9039  Good Fit


# **Final Model Training & Prediction**

In [91]:
best_model = Pipeline([
    ('pre', preprocessor),
    ('clf', LGBMClassifier(random_state=42))
])

In [92]:
cross_validation_score = np.mean(cross_val_score(best_model, X, y, cv=5, n_jobs=-1))

In [93]:
cross_validation_score

0.9044707058075938

In [95]:
# Fit on full training set
best_model.fit(X, y)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002100 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2035
[LightGBM] [Info] Number of data points in the train set: 20758, number of used features: 25
[LightGBM] [Info] Start training from score -2.107483
[LightGBM] [Info] Start training from score -1.907353
[LightGBM] [Info] Start training from score -1.964779
[LightGBM] [Info] Start training from score -1.854892
[LightGBM] [Info] Start training from score -1.635203
[LightGBM] [Info] Start training from score -2.146276
[LightGBM] [Info] Start training from score -2.107879


,steps,"[('pre', ...), ('clf', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('bin', ...), ('num', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


# **Predict on Test Data**

In [96]:
final_preds = best_model.predict(X_test_final)

d:\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


# **Final Submission**

In [97]:
# Decode back to original labels
final_preds_labels = le.inverse_transform(final_preds)

submission = pd.DataFrame({
    "id": test["id"],
    "NObeyesdad": final_preds_labels
})

submission.to_csv("submission.csv", index=False)
print("\n🎉 Submission file created: submission.csv")


🎉 Submission file created: submission.csv


# Check Hyper-Parameter Using GridsearchCV

In [100]:
# XGBoost Pipeline + GridSearch
# =========================
xgb_clf = XGBClassifier(use_label_encoder=False, eval_metric="mlogloss", random_state=42)

xgb_pipe = Pipeline([
    ("pre", preprocessor),
    ("clf", xgb_clf)
])

xgb_params = {
    "clf__n_estimators": [200, 400],
    "clf__max_depth": [3, 5, 7],
    "clf__learning_rate": [0.05, 0.1, 0.2],
    #"clf__subsample": [0.8, 1.0]
}

grid_xgb = run_search_grid(xgb_pipe, xgb_params, "XGBoost")



d:\anaconda3\Lib\site-packages\xgboost\training.py:183: UserWarning: [13:36:31] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



✅ Finished GridSearch for XGBoost
Best Params: {'clf__learning_rate': 0.2, 'clf__max_depth': 3, 'clf__n_estimators': 200}


In [103]:
# =========================
# LightGBM Pipeline + GridSearch
# =========================
lgbm_clf = LGBMClassifier(random_state=42)

lgbm_pipe = Pipeline([
    ("pre", preprocessor),
    ("clf", lgbm_clf)
])

lgbm_params = {
    "clf__n_estimators": [200, 400],
    "clf__max_depth": [-1, 5, 10],
    "clf__learning_rate": [0.05, 0.1, 0.2],
    #"clf__num_leaves": [31, 50, 100]
}

grid_lgb = run_search_grid(lgbm_pipe, lgbm_params, "LightGBM")



[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001471 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2031
[LightGBM] [Info] Number of data points in the train set: 16606, number of used features: 25
[LightGBM] [Info] Start training from score -2.117117
[LightGBM] [Info] Start training from score -1.911230
[LightGBM] [Info] Start training from score -1.948141
[LightGBM] [Info] Start training from score -1.857720
[LightGBM] [Info] Start training from score -1.633574
[LightGBM] [Info] Start training from score -2.145531
[LightGBM] [Info] Start training from score -2.112625
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with 

In [104]:
# =========================
# Evaluate Models
# =========================
results = []
grids = [
    ("XGBoost", grid_xgb),
    ("LightGBM", grid_lgb)
]

for name, grid in grids:
    res = evaluate_model(grid, name, X_train, y_train, X_valid, y_valid)
    results.append(res)

df_results = pd.DataFrame(results)
print("\n📊 Model Comparison:\n", df_results)



d:\anaconda3\Lib\site-packages\xgboost\training.py:183: UserWarning: [13:51:27] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
d:\anaconda3\Lib\site-packages\xgboost\training.py:183: UserWarning: [13:51:28] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
d:\anaconda3\Lib\site-packages\xgboost\training.py:183: UserWarning: [13:51:29] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
d:\anaconda3\Lib\site-packages\xgboost\training.py:183: UserWarning: [13:51:30] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
d:\anaconda3\Lib\sit


--- XGBoost ---
Validation Accuracy: 0.9079
Test Accuracy: 0.907
Gap: 0.0008
Fit Check: ✅ Good Fit
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000918 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2025
[LightGBM] [Info] Number of data points in the train set: 13284, number of used features: 25
[LightGBM] [Info] Start training from score -2.117182
[LightGBM] [Info] Start training from score -1.911577
[LightGBM] [Info] Start training from score -1.948397
[LightGBM] [Info] Start training from score -1.857563
[LightGBM] [Info] Start training from score -1.633359
[LightGBM] [Info] Start training from score -2.145728
[LightGBM] [Info] Start training from score -2.112191
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning

d:\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002236 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2033
[LightGBM] [Info] Number of data points in the train set: 13285, number of used features: 25
[LightGBM] [Info] Start training from score -2.116632
[LightGBM] [Info] Start training from score -1.911143
[LightGBM] [Info] Start training from score -1.948473
[LightGBM] [Info] Start training from score -1.858121
[LightGBM] [Info] Start training from score -1.633820
[LightGBM] [Info] Start training from score -2.145160
[LightGBM] [Info] Start training from score -2.112266
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with 

d:\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


...

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000907 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2030
[LightGBM] [Info] Number of data points in the train set: 13285, number of used features: 25
[LightGBM] [Info] Start training from score -2.117257
[LightGBM] [Info] Start training from score -1.911143
[LightGBM] [Info] Start training from score -1.947945
[LightGBM] [Info] Start training from score -1.857639
[LightGBM] [Info] Start training from score -1.633434
[LightGBM] [Info] Start training from score -2.145803
[LightGBM] [Info] Start training from score -2.112889
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with 

d:\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.003371 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2027
[LightGBM] [Info] Number of data points in the train set: 13285, number of used features: 25
[LightGBM] [Info] Start training from score -2.117257
[LightGBM] [Info] Start training from score -1.911143
[LightGBM] [Info] Start training from score -1.947945
[LightGBM] [Info] Start training from score -1.857639
[LightGBM] [Info] Start training from score -1.633434
[LightGBM] [Info] Start training from score -2.145803
[LightGBM] [Info] Start training from score -2.112889
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further s

d:\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
d:\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



--- LightGBM ---
Validation Accuracy: 0.9083
Test Accuracy: 0.9037
Gap: 0.0047
Fit Check: ✅ Good Fit

📊 Model Comparison:
       Model  Val_Accuracy  Test_Accuracy     Gap         Fit
0   XGBoost        0.9079         0.9070  0.0008  ✅ Good Fit
1  LightGBM        0.9083         0.9037  0.0047  ✅ Good Fit


In [106]:
# =========================
# Make Predictions & Submission
# =========================
best_lgb_model = grid_lgb.best_estimator_
final_preds = best_lgb_model.predict(X_test_final)

# Decode back to original labels
final_preds_labels = le.inverse_transform(final_preds)



d:\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [107]:
submission = pd.DataFrame({
    "id": test["id"],
    "NObeyesdad": final_preds_labels
})

submission.to_csv("submission1.csv", index=False)
print("\n🎉 Submission file created: submission.csv")


🎉 Submission file created: submission.csv
